# Notebook 06: Anomalias de Perfil e Caracterização das Faixas Etárias

**Unidade Curricular:** Dados e Aprendizagem Automática (DAA) — 2026/2027

Segunda metade da nota: usar o modelo para **detetar inconsistências de perfil** (idade declarada vs idade
prevista) e **caracterizar cada faixa etária** a nível socioeconómico e de saúde. Trabalha-se sobre as OOF do
melhor classificador puro do NB04 (sem escalas nem stacking: ambos distorcem priors e mascaram anomalias),
cruzadas com o texto do NB01. Sem treino — só análise.

## 0. Carga e medida de discrepância

Discrepância `d = prevista − real` em classes (1 classe ≈ 5 anos). `|d| >= 3` (≈15+ anos) define anomalia:
suficientemente longe para não ser erro off-by-one banal, suficientemente frequente (n≈466) para caracterizar.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('ggplot')
sns.set_theme(style='whitegrid')

DATA_DIR = '../data'
LIMIAR = 3

y = pd.read_csv(os.path.join(DATA_DIR, 'train_nb02.csv'), usecols=['PID', 'AGE_RANGE'],
                index_col='PID')['AGE_RANGE']
t1 = pd.read_csv(os.path.join(DATA_DIR, 'train_nb01.csv'), index_col='PID',
                 keep_default_na=False, na_values=[''])
z = np.load(os.path.join(DATA_DIR, 'nb04_oof.npz'))
pred = pd.Series(z['oof'], index=y.index)
d = (pred - y).rename('gap')
ad = d.abs()
print('exata:', round(float((ad == 0).mean()), 3), '| off-by-one:', round(float((ad == 1).mean()), 3))
print('anomalias |d|>=%d:' % LIMIAR, int((ad >= LIMIAR).sum()),
      '| >=4:', int((ad >= 4).sum()))

plt.figure(figsize=(9, 4))
sns.countplot(x=ad.clip(upper=6), hue=ad.clip(upper=6), legend=False, palette='viridis')
plt.title('Distribuicao de |prevista - real| (6+ agrupado)')
plt.xlabel('|gap| em classes')
plt.tight_layout()
plt.show()

## 1. Direções e controlo por routing

`gap > 0` = perfil envelhecido (prevista acima da real); `gap < 0` = rejuvenescido. Antes de chamar
"anomalia biológica" a qualquer caso, controla-se o routing: reformado previsto 15–17 ou estudante com 75–79
cheira a erro de registo, não a biologia. Comparam-se os dois sentidos com a população base em situação
profissional, sexo, comorbilidades e carga de routing.

In [ ]:
cd = [c for c in t1.columns if c.startswith('CD1')]
base = pd.DataFrame({'N_CD_SIM': (t1[cd] == 'Sim').sum(axis=1)})
base['N_NA'] = (t1.select_dtypes(include='str') == 'Não aplicável').sum(axis=1)
base['gap'] = d
base['grupo'] = np.where(ad >= LIMIAR, np.where(d > 0, 'envelhecida', 'rejuvenescida'), 'normal')
print(base['grupo'].value_counts().to_string())
print()
print(base.groupby('grupo')[['N_CD_SIM', 'N_NA']].mean().round(2).to_string())
print()
print('MAINSTAT das envelhecidas:')
print(t1.loc[base['grupo'] == 'envelhecida', 'MAINSTAT'].value_counts().head(6).to_string())
print('MAINSTAT das rejuvenescidas:')
print(t1.loc[base['grupo'] == 'rejuvenescida', 'MAINSTAT'].value_counts().head(6).to_string())

plt.figure(figsize=(10, 4))
sns.barplot(data=base.reset_index(), x='N_CD_SIM', y='grupo', errorbar='sd', orient='y')
plt.title('N comorbilidades por grupo')
plt.tight_layout()
plt.show()

## 2. Casos de estudo (para o relatório)

Os maiores gaps dos dois sentidos, com perfil compacto: situação profissional, estado civil, escolaridade,
saúde auto-percebida, nº de comorbilidades e lista das doenças declaradas. Servem a discussão erro-vs-biologia.

In [ ]:
COLS_CASO = ['MAINSTAT', 'MARSTALEGAL', 'HATLEVEL', 'HS1', 'SEX']
top = d.loc[ad >= LIMIAR].sort_values(key=abs, ascending=False).head(6)
for pid, gap in top.items():
    print('--- PID', pid, '| real', int(y.loc[pid]), '| prevista', int(pred.loc[pid]), '| gap', int(gap))
    print('   ', ' | '.join(str(t1.loc[pid, c])[:42] for c in COLS_CASO))
    print('    N_CD_SIM:', int(base.loc[pid, 'N_CD_SIM']), '| doencas:',
          [c for c in cd if t1.loc[pid, c] == 'Sim'][:8] or 'nenhuma')
    print()

## 3. Caracterização das 16 faixas (para o relatório)

Um retrato por faixa: dimensão, % mulheres, % empregados, % reformados, % viúvos, comorbilidades médias e
carga de routing. É a tabela-mãe da secção de caracterização; o heatmap mostra as transições de ciclo de vida
(estudo→emprego→reforma, solteiro→casado→viúvo) que o modelo aprendeu.

In [ ]:
g = y.to_frame().assign(Mulher=(t1['SEX'] == 'Mulher').values,
                     Empregada=(t1['MAINSTAT'].str.contains('Tem um emprego').fillna(False)).values,
                     Reformada=(t1['MAINSTAT'].str.contains('Reformado').fillna(False)).values,
                     Viuva=(t1['MARSTALEGAL'] == 'Viúvo').values,
                     N_CD_SIM=base['N_CD_SIM'].values,
                     N_NA=base['N_NA'].values).groupby('AGE_RANGE')
retrato = pd.DataFrame({'n': g.size(), '%Mulher': g['Mulher'].mean() * 100,
                        '%Emprego': g['Empregada'].mean() * 100, '%Reforma': g['Reformada'].mean() * 100,
                        '%Viuvo': g['Viuva'].mean() * 100, 'N_CD_medio': g['N_CD_SIM'].mean(),
                        'N_NA_medio': g['N_NA'].mean()}).round(1)
print(retrato.to_string())

plt.figure(figsize=(12, 5))
sns.heatmap(retrato[['%Mulher', '%Emprego', '%Reforma', '%Viuvo']].T, annot=True, fmt='.0f',
            cmap='viridis')
plt.title('Transicoes de ciclo de vida por faixa etaria (%)')
plt.tight_layout()
plt.show()

## Conclusões

- **Acerto exato 47.1%, off-by-one 38.1%; 466 anomalias (|d|>=3), 132 com gap>=4.** Pool suficiente para dois capítulos (207 envelhecidas, 259 rejuvenescidas).

- **Anomalias têm MENOS comorbilidades que o normal** (2.07/2.05 vs 2.58) e envelhecidas têm mais NA (68.9 vs 65.2): o mismatch é de routing/perfil socioprofissional, não de doença. Rejuvenescidas são 49% reformados (127/259) — velhos com perfil jovem.

- **Erro de registo vs biologia:** PID 12012 (75-79 estudante) e 14056 (15-17 casado) cheiram a erro de registo; 13301 (85+ empregado), 1265 (85+ zero doenças) e 5034 (45-49 estudante) são biologia plausível. É esta a distinção a argumentar no relatório.

- **Retrato por faixa (tabela-mãe):** emprego pika 86.7% aos 35-39; reforma vira 31%→72% entre 60-64 e 65-69; viuvez 0→63.5%; N_CD 0.5→4.4; NA cai aos 55-59 (entram as perguntas de autonomia).